#  EXP_001: Cálculo de π

En este "notebook" estudiaremos cómo de efectiva es la versión "especulativa" frente a la versión secuencial y la versión directamente paralelizada.
Nuestro primer paso es asegurarnos de que todo está preparado para ejecución. (Compilar y tratar las ENVs) y vamos a iniciar todas las variables de las que dependen la mayoria de nuestras celdas para evitar trabajar con valores distintos por error.

In [ ]:
import subprocess 

subprocess.run("/home/usc/cursos/curso1285/home/TFG/problemas/Problema_001/scripts/prepare_001.sh")

# Ejecutables
P001_SEC = "/home/usc/cursos/curso1285/home/TFG/problemas/Problema_001/execs/P_001_seq"
P001_OMP = "/home/usc/cursos/curso1285/home/TFG/problemas/Problema_001/execs/P_001_omp"
P001_ESP_1 = "/home/usc/cursos/curso1285/home/TFG/problemas/Problema_001/execs/P_001_esp_1"
P001_ESP_2 = "/home/usc/cursos/curso1285/home/TFG/problemas/Problema_001/execs/P_001_esp_2"

# CSVs
R001_SEC_CSV = "/home/usc/cursos/curso1285/home/TFG/problemas/Problema_001/results/P_001_sec.csv"
R001_OMP_CSV = "/home/usc/cursos/curso1285/home/TFG/problemas/Problema_001/results/P_001_omp.csv"
R001_ESP_1_CSV = "/home/usc/cursos/curso1285/home/TFG/problemas/Problema_001/results/P_001_esp_1.csv"
R001_ESP_2_CSV = "/home/usc/cursos/curso1285/home/TFG/problemas/Problema_001/results/P_001_esp_2.csv"

# CONTROL BUCLES
M = 10      # Numero de repeticiones de cada configuracion
N_VALUES = [int(1e5), int(1e6), int(1e7), int(1e8)]   # Tamaño del problema
T_VALUES = [2, 4, 6, 8]    # Numero de hilos
P_VALUES = [10]     # Porcentajes para especulativo 1


Primeramente, obtendremos resultados de la versión secuencial. Concretamente sobre su tiempo de ejecución y sobre su uso de memoria. (Media y desviación).

- **Tiempo**: Utilizamos el tiempo de ejecución de la cpu con la función clock().
- **Memoria**: Utilizamos el comando _/usr/bin/time_ para obtener el Maximo RSS (Maximo tamaño que el proceso ocupó en memoria).

In [31]:
import subprocess
import pandas as pd
import statistics
import csv

results_001_seq = []

# Computo del experimento.
for N in N_VALUES:
    times_001_seq = []
    rss_001_seq = []   
    for i in range(M):
        output = subprocess.run(
            ["/mnt/netapp1/Optcesga_FT2_RHEL7/2020/gentoo/22072020/usr/bin/time", "-v", P001_SEC, str(N)],
            capture_output=True,
            text=True
        )

        # Tiempo
        for line in output.stdout.splitlines():
            if line.startswith("TIME:"):
                times_001_seq.append(float(line.split("TIME:")[1].strip()))
                break
        # RSS
        for line in output.stderr.splitlines():
            if "Maximum resident set size (kbytes):" in line:
                rss_001_seq.append(float(line.split(":")[1].strip()))
                break

    times_001_seq_mean = sum(times_001_seq) / M
    rss_001_seq_mean = sum(rss_001_seq) / M
    
    results_001_seq.append({
        "N": N,
        "TIME_mean": times_001_seq_mean,
        "RSS_mean": rss_001_seq_mean,
        "TIME_std": statistics.stdev(times_001_seq),
        "RSS_std": statistics.stdev(rss_001_seq),
    })

# Pasamos los resultados a un csv
with open(R001_SEC_CSV, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["N", "TIME_mean", "RSS_mean", "TIME_std", "RSS_std"])
    
    writer.writeheader()   
    
    for row in results_001_seq:
        writer.writerow(row)

Una vez hemos obtenido los tiempos secuenciales, pasemos a la versión con puro OpenMP.

In [32]:
import subprocess
import pandas as pd
import statistics
import csv

results_001_omp = []

# Computo del experimento.
for N in N_VALUES:
    for T in T_VALUES:
        times_001_omp = []
        rss_001_omp = []   
        for i in range(M):
            output = subprocess.run(
                ["/mnt/netapp1/Optcesga_FT2_RHEL7/2020/gentoo/22072020/usr/bin/time", "-v", P001_OMP, str(N),str(T)],
                capture_output=True,
                text=True
            )

            # Tiempo
            for line in output.stdout.splitlines():
                if line.startswith("TIME:"):
                    times_001_omp.append(float(line.split("TIME:")[1].strip()))
                    break
            # RSS
            for line in output.stderr.splitlines():
                if "Maximum resident set size (kbytes):" in line:
                    rss_001_omp.append(float(line.split(":")[1].strip()))
                    break

            times_001_omp_mean = sum(times_001_omp) / M
            rss_001_omp_mean = sum(rss_001_omp) / M
            
        results_001_omp.append({
            "N": N,
            "T": T,
            "TIME_mean": times_001_omp_mean,
            "RSS_mean": rss_001_omp_mean,
            "TIME_std": statistics.stdev(times_001_omp),
            "RSS_std": statistics.stdev(rss_001_omp),
        })

    # Pasamos los resultados a un csv
    with open(R001_OMP_CSV, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["N", "T", "TIME_mean", "RSS_mean", "TIME_std", "RSS_std"])
        
        writer.writeheader()   
        
        for row in results_001_omp:
            writer.writerow(row)

KeyboardInterrupt: 

Ahora las versiones especulativas. De las versiones especualtivas pretendemos extraer más métricas interesantes:
- **Tiempo decision**: Cantidad de tiempo que el sistema invierte escogiendo la planificación más adecuada.

In [ ]:
import subprocess
import pandas as pd
import statistics
import csv

results_001_esp_1 = []

# Computo del experimento.
for N in N_VALUES:
    for T in T_VALUES:
        for p in P_VALUES:
            times_001_esp_1 = []
            times_dec_001_esp_1 = []
            rss_001_esp_1 = []   
            pred_S_001_esp_1 = 0
            for i in range(M):
                output = subprocess.run(
                    ["/mnt/netapp1/Optcesga_FT2_RHEL7/2020/gentoo/22072020/usr/bin/time", "-v", P001_ESP_1, str(N),str(T), str(p)],
                    capture_output=True,
                    text=True
                )

                # Tiempo Total
                for line in output.stdout.splitlines():
                    if line.startswith("TOTAL_TIME:"):
                        times_001_esp_1.append(float(line.split("TOTAL_TIME:")[1].strip()))
                        break
                
                # Tiempo de decision
                for line in output.stdout.splitlines():
                    if line.startswith("DECISSION_TIME:"):
                        times_dec_001_esp_1.append(float(line.split("DECISSION_TIME:")[1].strip()))
                        break
                
                # RSS
                for line in output.stderr.splitlines():
                    if "Maximum resident set size (kbytes):" in line:
                        rss_001_esp_1.append(float(line.split(":")[1].strip()))
                        break
                
                # Prediccion de planificación
                for line in output.stdout.splitlines():
                    if "WINNER:" in line:
                        if line.split(":")[1].strip() == 'S':
                            pred_S_001_esp_1 += 1
                        break
                times_001_esp_1_mean = sum(times_001_esp_1) / M
                times_dec_001_esp_1_mean = sum(times_dec_001_esp_1) / M
                rss_001_esp_1_mean = sum(rss_001_esp_1) / M
            
            results_001_esp_1.append({
                "N": N,
                "T": T,
                "p": p,
                "TIME_mean": times_001_esp_1_mean,
                "TIME_dec_mean": times_dec_001_esp_1_mean,
                "RSS_mean": rss_001_esp_1_mean,
                "TIME_std": statistics.stdev(times_001_esp_1),
                "RSS_std": statistics.stdev(rss_001_esp_1),
                "TIME_dec_std": statistics.stdev(times_dec_001_esp_1),
                "Pred_S": pred_S_001_esp_1
            })

    # Pasamos los resultados a un csv
    with open(R001_ESP_1_CSV, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["N", "T", "p", "TIME_mean", "RSS_mean", "TIME_dec_mean", "TIME_std", "RSS_std", "TIME_dec_std", "Pred_S"])
        writer.writeheader()   
        
        for row in results_001_esp_1:
            writer.writerow(row)




A continuación extraeremos los resultados de la versión especulativa 2:

In [ ]:
# TODO
# Insertar código aquí para extraer resultados.


A continación extraeremos métricas muy importantes como son el consumo de memoria en comparación con la versión secuencial y el SpeedUp.

In [ ]:
import pandas as pd
results_001_seq = pd.read_csv(R001_SEC_CSV)
results_001_omp = pd.read_csv(R001_OMP_CSV)

# Renombrar para evitar conflicto de nombres
results_001_seq = results_001_seq.rename(columns={"TIME_mean": "TIME_seq"})

# Hacer join por N
df = pd.merge(results_001_omp, results_001_seq[['N', 'TIME_seq']], on='N')

# Calcular speedup
df['speedup'] = df['TIME_seq'] / df['TIME_mean']

print(df)

print(df)

          N  T  TIME_mean  RSS_mean  TIME_std      RSS_std  TIME_seq   speedup
0        10  2   0.000058    2274.4  0.000042    78.705075  0.000002  0.036332
1        10  3   0.000053    2268.4  0.000003    71.986418  0.000002  0.039698
2        10  4   0.000076    2268.4  0.000006    45.233223  0.000002  0.027778
3        10  6   0.000152    2302.8  0.000070    92.280731  0.000002  0.013843
4        10  7   0.000171    2284.4  0.000064    69.010788  0.000002  0.012259
5        10  8   0.000194    2335.6  0.000070    87.136164  0.000002  0.010797
6      1000  2   0.000088    2329.6  0.000040    59.515077  0.000005  0.056625
7      1000  3   0.000115    2272.4  0.000042    73.210807  0.000005  0.043365
8      1000  4   0.000114    2255.6  0.000028    77.113481  0.000005  0.043668
9      1000  6   0.000150    2293.6  0.000043    59.395473  0.000005  0.033311
10     1000  7   0.000166    2257.2  0.000055    68.768210  0.000005  0.030084
11     1000  8   0.000167    2314.0  0.000012    75.